# Dual-MemoryLLM Authorship Verification Tutorial

This notebook demonstrates how to initialize, configure, and run a forward/backward pass using the **Dual-MemoryLLM** model for Authorship Verification.

We will demonstrate two variants:
1. **Unigram Variant**: No Phrase-Level memory. Extracts features directly from token-level initial embeddings.
2. **N-Gram Variant**: The full model utilizing all 3 pathways (Contextual, Phrase-Level N-grams, Token-Level Memory).

Both variants will be initialized from `answerdotai/ModernBERT-base` using `from_pretrained()`.

In [1]:
import torch
from modeling_dual_memoryllm import DualMemoryLLMConfig, DualMemoryLLMForAuthorshipVerification

model_id = "answerdotai/ModernBERT-base"

Skipping import of cpp extensions due to incompatible torch version 2.10.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


### 1. Unigram Variant (use_ngram_memory=False)
Initialize Dual-MemoryLLM bypassing N-gram tables.

In [2]:
print("=== Initializing Unigram Variant ===")
config_unigram = DualMemoryLLMConfig.from_pretrained(model_id)

# Configure for downstream logic
config_unigram.use_ngram_memory = False

# We ignore mismatched sizes during loading since we append our own custom heads and pathways
model_unigram = DualMemoryLLMForAuthorshipVerification.from_pretrained(
    model_id, 
    config=config_unigram, 
    ignore_mismatched_sizes=True
)

# Mock input batch [Batch_Size, Sequence_Length]
input_ids = torch.randint(1, 100, (2, 32)) 
outputs_unigram = model_unigram(input_ids=input_ids)

print("--- Forward Pass Results ---")
print(f"Final Authorship Representation Matrix: {outputs_unigram['authorship_representation'].shape}")
print(f"Top-K Engram Fingerprint Indices Shape: {outputs_unigram['topk_indices'].shape}")

=== Initializing Unigram Variant ===


Some weights of DualMemoryLLMForAuthorshipVerification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['model.layers.0.token_memory_norm.weight', 'model.layers.1.token_memory_norm.weight', 'model.layers.10.token_memory_norm.weight', 'model.layers.11.token_memory_norm.weight', 'model.layers.12.token_memory_norm.weight', 'model.layers.13.token_memory_norm.weight', 'model.layers.14.token_memory_norm.weight', 'model.layers.15.token_memory_norm.weight', 'model.layers.16.token_memory_norm.weight', 'model.layers.17.token_memory_norm.weight', 'model.layers.18.token_memory_norm.weight', 'model.layers.19.token_memory_norm.weight', 'model.layers.2.token_memory_norm.weight', 'model.layers.20.token_memory_norm.weight', 'model.layers.21.token_memory_norm.weight', 'model.layers.3.token_memory_norm.weight', 'model.layers.4.token_memory_norm.weight', 'model.layers.5.token_memory_norm.weight', 'model.layers.6.token_memory_norm.weight', 'model.lay

--- Forward Pass Results ---
Final Authorship Representation Matrix: torch.Size([2, 32, 768])
Top-K Engram Fingerprint Indices Shape: torch.Size([2, 32])


### 2. N-Gram Variant (use_ngram_memory=True)
Initialize the full Dual-MemoryLLM with Phrase-Level Memory.

In [3]:
print("=== Initializing Full N-Gram Variant ===")
config_ngram = DualMemoryLLMConfig.from_pretrained(model_id)

# Engram specialized configuration
config_ngram.use_ngram_memory = True
config_ngram.engram_layer_ids = [1, 3]          # which layers inject Phrase-Level Memory
config_ngram.engram_vocab_size = [1000, 1000]   # size of the n-gram hashing vocab tables
config_ngram.max_ngram_size = 3
config_ngram.n_embed_per_ngram = 64
config_ngram.n_head_per_ngram = 2

# We ignore mismatched sizes during loading since we append our own custom heads and pathways
model_ngram = DualMemoryLLMForAuthorshipVerification.from_pretrained(
    model_id, 
    config=config_ngram, 
    ignore_mismatched_sizes=True
)

outputs_ngram = model_ngram(input_ids=input_ids)

print("--- Forward Pass Results ---")
print(f"Final Authorship Representation Matrix: {outputs_ngram['authorship_representation'].shape}")
print(f"Top-K Engram Fingerprint Indices Shape: {outputs_ngram['topk_indices'].shape}")

=== Initializing Full N-Gram Variant ===


Some weights of DualMemoryLLMForAuthorshipVerification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['model.layers.0.token_memory_norm.weight', 'model.layers.1.token_memory_norm.weight', 'model.layers.10.token_memory_norm.weight', 'model.layers.11.token_memory_norm.weight', 'model.layers.12.token_memory_norm.weight', 'model.layers.13.token_memory_norm.weight', 'model.layers.14.token_memory_norm.weight', 'model.layers.15.token_memory_norm.weight', 'model.layers.16.token_memory_norm.weight', 'model.layers.17.token_memory_norm.weight', 'model.layers.18.token_memory_norm.weight', 'model.layers.19.token_memory_norm.weight', 'model.layers.2.token_memory_norm.weight', 'model.layers.20.token_memory_norm.weight', 'model.layers.21.token_memory_norm.weight', 'model.layers.3.token_memory_norm.weight', 'model.layers.4.token_memory_norm.weight', 'model.layers.5.token_memory_norm.weight', 'model.layers.6.token_memory_norm.weight', 'model.lay

--- Forward Pass Results ---
Final Authorship Representation Matrix: torch.Size([2, 32, 768])
Top-K Engram Fingerprint Indices Shape: torch.Size([2, 32])


### 3. Demonstrating End-to-End Gradients
We demonstrate that both variants correctly participate in the backwards routing graph.

In [4]:
print("=== Backward Pass Checks ===")
loss_unigram = outputs_unigram["authorship_representation"].sum()
loss_unigram.backward()

has_attn_grad_u = model_unigram.model.layers[0].attn.Wqkv.weight.grad is not None
has_token_mem_grad_u = model_unigram.model.layers[0].token_memory_mlp.Wi.weight.grad is not None
print(f"Unigram Pathways received gradients: Contextual: {has_attn_grad_u}, Token-Level: {has_token_mem_grad_u}")

loss_ngram = outputs_ngram["authorship_representation"].sum()
loss_ngram.backward()

has_attn_grad_ng = model_ngram.model.layers[0].attn.Wqkv.weight.grad is not None
has_token_mem_grad_ng = model_ngram.model.layers[0].token_memory_mlp.Wi.weight.grad is not None
has_engram_grad_ng = model_ngram.model.layers[1].engram.value_proj.weight.grad is not None
print(f"N-Gram Pathways received gradients: Contextual: {has_attn_grad_ng}, Phrase-Level: {has_engram_grad_ng}, Token-Level: {has_token_mem_grad_ng}")

=== Backward Pass Checks ===
Unigram Pathways received gradients: Contextual: True, Token-Level: True


N-Gram Pathways received gradients: Contextual: True, Phrase-Level: True, Token-Level: True
